# Output schemas for point data

This notebook shows how to use the `weather_tools` output-schema module when:

- **Consuming** data returned by the API/local functions (validate it conforms to the contract)
- **Integrating** with a downstream package (use the schema as a reference for expected columns)

## Sources covered

| Source | Function | Schema |
|--------|----------|--------|
| SILO PatchedPoint | `SiloAPI.get_patched_point()` | `SiloPointSchema` |
| SILO DataDrill | `SiloAPI.get_data_drill()` | `SiloPointSchema` |
| met.no forecast | `MetNoAPI.get_daily_forecast()` | `MetNoForecastSchema` |
| Merged history+forecast | `merge_historical_and_forecast()` | `MergedPointSchema` |

**Prerequisites**: set `SILO_API_KEY` to your registered SILO e-mail address.

In [ ]:
import os

import pandas as pd

from weather_tools.merge_weather_data import merge_historical_and_forecast
from weather_tools.metno_api import MetNoAPI
from weather_tools.output_schemas import (
    DATE_COLUMN,
    MetNoForecastSchema,
    MergedPointSchema,
    PointMetadata,
    SiloPointSchema,
    validate_point_dataframe,
)
from weather_tools.silo_api import SiloAPI

print(f"Standard date column name: '{DATE_COLUMN}'")

## Helper: preparing a raw SILO response for schema validation

The SILO API attaches several implementation-specific columns to raw responses that are
not part of the point-data contract:

| Column pattern | Source | How to handle |
|---|---|---|
| `*_source` (e.g. `daily_rain_source`) | Data quality flags | Drop before schema validation |
| `station` | PatchedPoint location | Pass to `PointMetadata.station_code` |
| `latitude` / `longitude` | DataDrill location | Pass to `PointMetadata` |
| `metadata` | Query metadata (row 0 only) | Use `return_metadata=True` instead |

The helper below strips these so the DataFrame matches the schema contract.

In [ ]:
def prepare_silo_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop SILO-specific extra columns that are not part of the point-data schema.

    Strips: *_source quality flags, metadata row, and location columns
    (station, latitude, longitude) that belong in PointMetadata instead.
    """
    drop = [
        c
        for c in df.columns
        if c.endswith("_source")
        or c in ("metadata", "station", "latitude", "longitude")
    ]
    return df.drop(columns=drop, errors="ignore")

## 1. SILO PatchedPoint → `SiloPointSchema`

PatchedPoint returns station observational data with infilled gaps.

In [ ]:
STATION_CODE = "40913"  # Brisbane
VARIABLES = ["daily_rain", "max_temp", "min_temp", "radiation", "vp"]
START = "20250101"
END = "20250107"

silo = SiloAPI()
raw_pp, pp_meta = silo.get_patched_point(
    station_code=STATION_CODE,
    start_date=START,
    end_date=END,
    variables=VARIABLES,
    return_metadata=True,
)

print("Raw PatchedPoint columns:", raw_pp.columns.tolist())
print("\nMetadata dict:", pp_meta)
raw_pp.head()

In [ ]:
# Prepare and validate
pp_df = prepare_silo_df(raw_pp)
print("Schema-ready columns:", pp_df.columns.tolist())

issues = validate_point_dataframe(pp_df, SiloPointSchema)
print("\nValidation issues:", issues or "none — DataFrame is schema-conformant")

pp_df.head()

In [ ]:
# Pair the DataFrame with a PointMetadata companion
import datetime as dt

pp_metadata = PointMetadata(
    latitude=-27.481,  # Brisbane station lat
    longitude=153.039,
    station_code=pp_meta["station_code"],
    source="silo_patched_point",
    start_date=dt.date(2025, 1, 1),
    end_date=dt.date(2025, 1, 7),
    variables=pp_meta["variables"],
)

print(pp_metadata.model_dump())

## 2. SILO DataDrill → `SiloPointSchema`

DataDrill returns gridded interpolated data for any lat/lon coordinate.

In [ ]:
LAT, LON = -27.5, 153.0

raw_dd, dd_meta = silo.get_data_drill(
    latitude=LAT,
    longitude=LON,
    start_date=START,
    end_date=END,
    variables=VARIABLES,
    return_metadata=True,
)

print("Raw DataDrill columns:", raw_dd.columns.tolist())
raw_dd.head()

In [ ]:
dd_df = prepare_silo_df(raw_dd)

issues = validate_point_dataframe(dd_df, SiloPointSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

dd_df.head()

## 3. met.no forecast → `MetNoForecastSchema`

Met.no daily forecasts use **descriptive column names** that differ from SILO canonical
names. The schema documents the mapping.

| SILO name | Met.no name |
|---|---|
| `daily_rain` | `total_precipitation` |
| `max_temp` | `max_temperature` |
| `min_temp` | `min_temperature` |
| `vp` | `avg_relative_humidity` (converted) |
| `mslp` | `avg_pressure` |

In [ ]:
metno = MetNoAPI()
forecast_df = metno.get_daily_forecast(latitude=LAT, longitude=LON, days=7)

print("Forecast columns:", forecast_df.columns.tolist())
print(f"\nDate column dtype: {forecast_df['date'].dtype}")
forecast_df

In [ ]:
# No preparation needed — get_daily_forecast already returns a clean DataFrame
issues = validate_point_dataframe(forecast_df, MetNoForecastSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

## 4. Merged history + forecast → `MergedPointSchema`

`merge_historical_and_forecast()` converts met.no columns to SILO canonical names
and adds provenance columns (`data_source`, `is_forecast`).

In [ ]:
# Pull SILO history for the week immediately before the forecast window
forecast_start = forecast_df["date"].min()
history_end = forecast_start - pd.Timedelta(days=1)
history_start = history_end - pd.Timedelta(days=6)

silo_history = silo.get_data_drill(
    latitude=LAT,
    longitude=LON,
    start_date=history_start.strftime("%Y%m%d"),
    end_date=history_end.strftime("%Y%m%d"),
    variables=["daily_rain", "max_temp", "min_temp"],
)

merged_df = merge_historical_and_forecast(
    silo_data=silo_history,
    metno_data=forecast_df,
    overlap_strategy="prefer_silo",
)

print("Merged columns:", merged_df.columns.tolist())
merged_df[["date", "daily_rain", "max_temp", "min_temp", "data_source", "is_forecast"]]

In [ ]:
issues = validate_point_dataframe(merged_df, MergedPointSchema)
print("Validation issues:", issues or "none — DataFrame is schema-conformant")

## 5. Using schemas as an integration reference

`SiloPointSchema.column_descriptions()` returns a machine-readable dictionary that
downstream packages can use to discover available variables and their units.

In [ ]:
descriptions = SiloPointSchema.column_descriptions()

# Formatted for reading
print(f"{'Column':<25} {'Description'}")
print("-" * 65)
for col, desc in descriptions.items():
    if desc:  # skip date which has no unit annotation
        print(f"{col:<25} {desc}")

In [ ]:
# Downstream packages can also inspect required columns programmatically
print("SiloPointSchema required columns :", SiloPointSchema.REQUIRED_COLUMNS)
print("MetNoForecastSchema required columns:", MetNoForecastSchema.REQUIRED_COLUMNS)
print("MergedPointSchema required columns  :", MergedPointSchema.REQUIRED_COLUMNS)

## 6. Module constants for safe column access

Use the exported constants instead of hard-coded strings to avoid typos.

In [ ]:
from weather_tools.output_schemas import (
    DATA_SOURCE_COLUMN,
    DATE_COLUMN,
    IS_FORECAST_COLUMN,
)

# Using constants when slicing DataFrames prevents silent bugs if column names change
forecast_rows = merged_df[merged_df[IS_FORECAST_COLUMN] == True]
silo_rows = merged_df[merged_df[DATA_SOURCE_COLUMN] == "silo"]

print(f"Forecast rows  ({IS_FORECAST_COLUMN}=True): {len(forecast_rows)}")
print(f"Historical rows ({DATA_SOURCE_COLUMN}='silo'): {len(silo_rows)}")
print(f"\nDate range via constant: {merged_df[DATE_COLUMN].min().date()} → {merged_df[DATE_COLUMN].max().date()}")